# Caching & Persistence

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

Three claims under test, each checked directly against real evidence rather than assumed:
1. `.cache()` + a forcing action makes the Storage tab / `/api/v1/applications/<id>/storage/rdd` show a real fraction cached, in memory vs. on disk.
2. Repeated actions against a cached DataFrame are measurably faster than repeated actions against a comparable-cost, never-cached DataFrame -- this notebook measures and prints the actual speedup ratio, it does not assume a number.
3. `MEMORY_ONLY` vs `MEMORY_AND_DISK` behave differently once the cached data exceeds available cache memory: `MEMORY_ONLY` drops the overflow (fraction cached < 100%, no disk usage); `MEMORY_AND_DISK` spills it (fraction cached at 100%, nonzero disk usage).

In [ ]:
import json
import time
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark import StorageLevel
from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("caching-persistence").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def storage_snapshot():
    """Current `/api/v1/applications/<id>/storage/rdd` list -- the ground
    truth for 'what fraction of this DataFrame is cached, and where', the
    same REST surface the Spark UI's Storage tab itself reads. No fetcher for
    this endpoint exists in app/spark_api/app_client.py (checked: its
    fetchers cover jobs/stages/executors/tasks only) and the annotation
    manifest schema has no storage-shaped rule type -- same disposition as
    dag-lazy-evaluation's jobs_snapshot(), queried directly here."""
    url = f"http://localhost:4040/api/v1/applications/{app_id}/storage/rdd"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


SCRATCH_DIR = "/workspace/scratch/caching-persistence"

## 1. Build an expensive DataFrame -- a join of two large parquet tables

Same real-file-backed pattern as `join-strategies`: write to parquet and read back so the join is a genuine shuffle over real files, not an in-memory toy.

In [ ]:
# Warm up the JVM/Catalyst codegen path with a throwaway job first -- otherwise the
# *first* real action below pays one-time class-loading/codegen cost that has nothing
# to do with caching, which would make the recompute-vs-cache-read comparison unfair.
spark.range(10).count()

FACT_ROWS = 1_000_000
DIM_ROWS = 400_000
NUM_KEYS = DIM_ROWS

fact_rdd = spark.sparkContext.parallelize(range(FACT_ROWS), 24).map(
    lambda i: Row(id=i % NUM_KEYS, amount=float(i % 997))
)
dim_rdd = spark.sparkContext.parallelize(range(DIM_ROWS), 24).map(
    lambda i: Row(id=i, category=f"cat-{i % 50}")
)

spark.createDataFrame(fact_rdd).write.mode("overwrite").parquet(f"{SCRATCH_DIR}/fact")
spark.createDataFrame(dim_rdd).write.mode("overwrite").parquet(f"{SCRATCH_DIR}/dim")
fact_df = spark.read.parquet(f"{SCRATCH_DIR}/fact")
dim_df = spark.read.parquet(f"{SCRATCH_DIR}/dim")

# Disable broadcast so this genuinely shuffles both sides (a sort-merge join) --
# the expensive step caching is meant to avoid repeating.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "12")

import math
from pyspark.sql.types import DoubleType

@F.udf(DoubleType())
def _expensive_score(amount):
    """A deliberately slow, row-at-a-time Python UDF -- found by running this notebook
    for real against this local dev cluster: a plain hash join, even over tens of
    millions of rows, finishes in well under a second on fast local disk + loopback
    networking, too fast for a cached read vs. full recompute to show any real
    difference. A row-at-a-time Python UDF's per-row interpreter + serialization
    overhead is genuinely expensive (the same real cost the UDF-vs-pandas-UDF topic
    covers directly) -- a cached DataFrame skips recomputing it entirely (reads the
    already-computed values back); an uncached DataFrame re-runs it, row by row, on
    every action."""
    x = amount
    for _ in range(30):
        x = math.sqrt(abs(math.sin(x) + math.cos(x)))
    return x


def with_expensive_score(df):
    return df.withColumn("score", _expensive_score(F.col("amount")))

print("fact_df partitions:", fact_df.rdd.getNumPartitions(), " dim_df partitions:", dim_df.rdd.getNumPartitions())

## 2. `.cache()` + `.count()` -- confirm the Storage tab shows a real fraction cached

**Hypothesis:** before the forcing `.count()`, does the storage endpoint show any entry for this DataFrame at all? What do you expect `numCachedPartitions` / `numPartitions` and `memoryUsed` / `diskUsed` to look like right after it runs?

In [ ]:
cached_df = with_expensive_score(fact_df.join(dim_df, on="id", how="inner")).cache()

before = storage_snapshot()
print(f"Storage entries before materialization: {len(before)}")
assert len(before) == 0, "expected nothing cached yet -- .cache() is lazy, same as any transformation"

t0 = time.perf_counter()
row_count = cached_df.count()
first_action_s = time.perf_counter() - t0
print(f"cached_df.count() = {row_count} rows, first action took {first_action_s:.2f}s")

after = storage_snapshot()
assert len(after) == 1, f"expected exactly one cached RDD entry, got {len(after)}"
entry = after[0]
fraction_cached = entry["numCachedPartitions"] / entry["numPartitions"]
print(f"Storage level: {entry['storageLevel']}")
print(f"Fraction cached: {entry['numCachedPartitions']}/{entry['numPartitions']} = {fraction_cached:.0%}")
print(f"Memory used: {entry['memoryUsed'] / 1e6:.1f} MB, Disk used: {entry['diskUsed'] / 1e6:.1f} MB")

In [ ]:
checkpoint(cached_df, topic="caching-persistence")
print("Checkpoint written -- click 'Reveal self-check' on the topic page: the plan now shows "
      "InMemoryTableScan feeding the join, not a repeated source scan.")

## 3. Timing: cached repeats vs. a single-use, never-cached DataFrame

`cached_df`'s first action above already paid the full join cost once (Storage tab confirmed above). Its 2nd/3rd actions should read straight from storage. `uncached_df` below is a fresh, comparable-cost join that is *never* cached -- each action against it repeats the whole shuffle from scratch, same as `cached_df`'s first action did.

**Hypothesis:** will `cached_df`'s 2nd/3rd actions be faster, the same, or slower than its 1st? Will `uncached_df`'s 2nd/3rd actions be any faster than its 1st?

In [ ]:
def timed_materialize(df):
    """agg(sum("score")), not count() or foreach() -- found by running this notebook for
    real: .count() lets Catalyst's column-pruning optimizer skip evaluating "score" entirely
    (it only needs row counts). foreach(lambda row: None) forces every column to be read, but
    routes every row through a Python worker for the no-op lambda itself, which is expensive
    regardless of caching and swamps the real signal. Aggregating "score" forces the actual
    value to be produced without that extra per-row Python round-trip."""
    t0 = time.perf_counter()
    df.agg(F.sum("score")).collect()
    return time.perf_counter() - t0

cached_times = [first_action_s]
for _ in range(2):
    elapsed = timed_materialize(cached_df)
    cached_times.append(elapsed)
print("cached_df action times (s):", [f"{t:.2f}" for t in cached_times])

# uncached_df is built from a SEPARATE, freshly-generated source (fact2/dim2, same size and
# shape as fact_df/dim_df, just different values) -- not a fresh join over the SAME fact_df/
# dim_df. Found by running this notebook for real: Spark's cache manager automatically
# substitutes cached_df's already-computed result into *any* later query whose analyzed plan
# is structurally identical (same relations, same transforms) to a cached one, even without
# ever calling .cache() on that later DataFrame -- so `fact_df.join(dim_df, ...)` built again
# here would have silently ridden on cached_df's cache the whole time, making "uncached" look
# just as fast as cached for the wrong reason. A genuinely different source table sidesteps
# that plan-matching and forces real, comparable-cost recomputation on every action.
fact2_rdd = spark.sparkContext.parallelize(range(FACT_ROWS), 24).map(
    lambda i: Row(id=i % NUM_KEYS, amount=float((i + 1) % 997))
)
dim2_rdd = spark.sparkContext.parallelize(range(DIM_ROWS), 24).map(
    lambda i: Row(id=i, category=f"cat-{(i + 1) % 50}")
)
spark.createDataFrame(fact2_rdd).write.mode("overwrite").parquet(f"{SCRATCH_DIR}/fact2")
spark.createDataFrame(dim2_rdd).write.mode("overwrite").parquet(f"{SCRATCH_DIR}/dim2")
fact2_df = spark.read.parquet(f"{SCRATCH_DIR}/fact2")
dim2_df = spark.read.parquet(f"{SCRATCH_DIR}/dim2")

uncached_df = with_expensive_score(fact2_df.join(dim2_df, on="id", how="inner"))  # never cached
uncached_times = []
for _ in range(3):
    elapsed = timed_materialize(uncached_df)
    uncached_times.append(elapsed)
print("uncached_df action times (s):", [f"{t:.2f}" for t in uncached_times])

cached_repeat_avg = sum(cached_times[1:]) / len(cached_times[1:])
uncached_avg = sum(uncached_times) / len(uncached_times)
speedup = uncached_avg / cached_repeat_avg
print(f"\ncached_df repeat-action avg: {cached_repeat_avg:.3f}s")
print(f"uncached_df action avg:      {uncached_avg:.3f}s")
print(f"Measured speedup from caching: {speedup:.1f}x")
assert cached_repeat_avg < uncached_avg, "expected cached repeats to be faster than the never-cached DataFrame"
cached_df.unpersist(blocking=True)  # release before section 4's storage-level demo

## 4. Storage levels under memory pressure: `MEMORY_ONLY` vs `MEMORY_AND_DISK`

A wide DataFrame (many double columns) sized to exceed this cluster's default per-executor cache capacity (executors here default to Spark's 1GB `spark.executor.memory`, unset by this project's compose template -- so the whole cluster's storage pool is only a few hundred MB, easy to exceed deliberately). Same row-generation cost both times; only the storage level differs.

**Hypothesis:** with `MEMORY_ONLY`, do you expect 100% cached, or something less? What about disk usage? Then, with `MEMORY_AND_DISK` on the same data -- does fraction cached change? Does disk usage?

In [ ]:
# Sized to exceed this cluster's total cache capacity (~1.3GB across 3 executors'
# default 1GB heaps, confirmed by querying /executors). Built via spark.range() +
# column expressions (fully vectorized, JVM-native codegen) rather than
# rdd.map(make_wide_row) -- found by running this notebook for real: the
# Python-Row-object construction approach OOM'd executors outright (Java heap
# space, executors lost, job aborted) even after repeated row-count/partition
# reductions -- per-row Python object construction plus py4j serialization for
# 20M rows costs orders of magnitude more CPU/memory than Spark SQL's native
# columnar path, and it never got far enough to demonstrate eviction at all.
# The vectorized version below computes the identical values (same
# `(i * 999983 + j * 7919) % 1_000_003` formula, now a column expression) at a
# fraction of the cost. High-cardinality values (a large prime modulus) keep
# the columnar cache from compressing this away via dictionary/RLE encoding,
# so the total genuinely doesn't fit in ~1.3GB of cache.
WIDE_ROWS = 16_000_000
NUM_DOUBLE_COLS = 20
WIDE_PARTITIONS = 96

wide_df = spark.range(WIDE_ROWS, numPartitions=WIDE_PARTITIONS).select(
    F.col("id"),
    *[
        ((F.col("id") * 999983 + F.lit(j * 7919)) % 1_000_003)
        .cast("double")
        .alias(f"c{j}")
        for j in range(NUM_DOUBLE_COLS)
    ],
)


def persist_and_measure(level, level_name):
    df = wide_df.persist(level)
    df.count()
    snap = storage_snapshot()
    assert len(snap) == 1, f"expected exactly one cached RDD entry, got {len(snap)}"
    entry = snap[0]
    fraction = entry["numCachedPartitions"] / entry["numPartitions"]
    print(f"[{level_name}] storageLevel={entry['storageLevel']}")
    print(f"[{level_name}] fraction cached: {entry['numCachedPartitions']}/{entry['numPartitions']} = {fraction:.0%}")
    print(f"[{level_name}] memoryUsed={entry['memoryUsed'] / 1e6:.1f} MB diskUsed={entry['diskUsed'] / 1e6:.1f} MB")
    df.unpersist(blocking=True)
    return fraction, entry["diskUsed"]

memory_only_fraction, memory_only_disk = persist_and_measure(StorageLevel.MEMORY_ONLY, "MEMORY_ONLY")
print()
memory_and_disk_fraction, memory_and_disk_disk = persist_and_measure(StorageLevel.MEMORY_AND_DISK, "MEMORY_AND_DISK")

print(f"\nMEMORY_ONLY:      fraction cached {memory_only_fraction:.0%}, disk used {memory_only_disk / 1e6:.1f} MB")
print(f"MEMORY_AND_DISK:   fraction cached {memory_and_disk_fraction:.0%}, disk used {memory_and_disk_disk / 1e6:.1f} MB")
assert memory_and_disk_fraction >= memory_only_fraction, (
    "expected MEMORY_AND_DISK to cache at least as large a fraction as MEMORY_ONLY (it spills the "
    "overflow instead of dropping it)"
)

## 5. Clean up

Release the caches this notebook built and reset the session-level conf it changed.

In [ ]:
cached_df.unpersist(blocking=True)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)